### DoubleDQN vs Random
Analyzing results of 15k episode training. Note: max steps per episode was 200 and max steps without pawn movement or capture was 20 and 300k steps of epsilon decay (steps of agent, which decayed from 1.0 to 0.05 around 11500th episode), pieces on corners were random Knight/Bishop/Rook and promotion choices were: Knight, Bishop, Rook and Queen.

In [1]:
from collections import Counter
from environment.grenight_environment import GrenightEnvironment
from agents.double_dqn_vs_random.agent import DoubleDQNAgent
import torch

In [2]:
def play_agent_vs_random_game(env_arg: GrenightEnvironment,
                              agent: DoubleDQNAgent,
                              agent_is_white: bool = True) -> tuple[str, int,  dict]:
    state = env_arg.reset()
    done = False
    move_count = 0
    reward = 0.0
    info = None

    while not done and move_count < 200:

        acting_player_is_white = env_arg.is_white_on_turn
        legal_mask = env_arg.action_mask()

        if acting_player_is_white == agent_is_white:
            action = agent.select_action(
                state,
                legal_mask,
                acting_player_is_white,
                epsilon=0.0,
            )
        else:
            action = env_arg.sample()

        state, reward, done, info = env_arg.step(action)
        move_count += 1

    if not done:
        return "truncated", move_count, info

    if reward == 0.0:
        return "draw", move_count, info

    winner_is_white = (
        acting_player_is_white
        if reward > 0
        else not acting_player_is_white
    )

    if winner_is_white == agent_is_white:
        return "agent_win", move_count, info
    else:
        return "random_win", move_count, info

In [3]:
env = GrenightEnvironment()

agent = DoubleDQNAgent(
    num_planes=env.state_encoder.NUM_PLANES,
    rows=5,
    columns=4,
    num_actions=env.action_encoder.NUM_ACTIONS,
    device="cpu",
)

checkpoint = torch.load(
    "../double_dqn_vs_random/ep15000.pt",
    map_location="cpu"
)

agent.policy_net.load_state_dict(checkpoint["policy_state_dict"])
agent.target_net.load_state_dict(checkpoint["target_state_dict"])

agent.policy_net.eval()
agent.target_net.eval()

print("Loaded checkpoint from episode:", checkpoint["episode"])
print("Global step:", checkpoint["global_step"])

outcomes_counter = Counter()
total_step_counts_per_outcome = Counter()
draw_reasons = Counter()

for _ in range(500):
    outcome, steps, game_info = play_agent_vs_random_game(env, agent, agent_is_white=True)
    outcomes_counter[outcome] += 1
    if outcome != "truncated":
        total_step_counts_per_outcome[outcome] += steps
    if game_info["draw_reason"] is not None:
        draw_reasons[game_info["draw_reason"]] += 1

avg_step_counts_per_outcome = dict()
for outcome, total_steps in total_step_counts_per_outcome.items():
    avg_step_counts_per_outcome[outcome] = total_steps / outcomes_counter[outcome]

print(f"Outcomes in 500 games where max steps without pawn movement or capturing: 20\n"
      f"And total max steps per episode was 200:\n"
      f"{outcomes_counter}\n"
      f"Average moves per outcome: \n{avg_step_counts_per_outcome}\n"
      f"Draw reasons: \n{draw_reasons}\n")

Loaded checkpoint from episode: 15000
Global step: 824240
Outcomes in 500 games where max steps without pawn movement or capturing: 20
And total max steps per episode was 200:
Counter({'draw': 417, 'agent_win': 61, 'random_win': 21, 'truncated': 1})
Average moves per outcome: 
{'agent_win': 45.295081967213115, 'draw': 76.71223021582733, 'random_win': 32.476190476190474}
Draw reasons: 
Counter({'stalemate': 168, 'threefold_repetition': 92, 'insufficient_material': 80, 'max_steps_without_progress': 77})

